# VIAGO V2 assessment assembly research

## TL;DR
The reproducible simulator compares the production-shaped baseline with three non-production architectures and three legacy-admission policies. The decision artifact is `docs/v2/ASSESSMENT_ASSEMBLY_RESEARCH.md`; this notebook is a compact reproducibility wrapper around the saved JSON evidence.

## Context and method
The bank is the preserved 151 active legacy questions plus 55 OWNER-reviewed, non-production development candidates. Each scenario contains 5,000 deterministic attempts. No production database or runtime endpoint is queried or mutated.

In [ ]:
import json
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()
simulation = json.loads((root / 'data/v2-research/assessment-assembler-simulation.json').read_text())
examples = json.loads((root / 'data/v2-research/example-attempt-manifests.json').read_text())
simulation['bank_profile'], len(examples['manifests'])

In [ ]:
rows = []
for scenario in simulation['scenarios']:
    rows.append({
        'scenario': scenario['scenario_id'],
        'attempts': scenario['iterations'],
        'question_overlap': scenario['pairwise_overlap']['total']['mean'],
        'single_overlap': scenario['pairwise_overlap']['single']['mean'],
        'likert_overlap': scenario['pairwise_overlap']['likert']['mean'],
        'semantic_overlap': scenario['pairwise_overlap']['semantic_family']['mean'],
        'domains': scenario['within_attempt']['unique_domains']['mean'],
        'contexts': scenario['within_attempt']['unique_contexts']['mean'],
        'workplace': scenario['within_attempt']['workplace_questions']['mean'],
        'weak_legacy': scenario['within_attempt']['weak_legacy_questions']['mean'],
    })
rows

## Results and takeaways
Design B with capped legacy admission is the recommended research configuration: equal Likert opportunity, a near-current 24/26 response-format mix, bounded weak/workplace exposure, zero within-attempt semantic collisions, and materially lower repeat overlap. Strict weak-item exclusion is not recommended yet because the smaller eligible bank increases repeat overlap. No scoring or production authority follows from this simulation.

In [ ]:
baseline = next(row for row in rows if row['scenario'] == 'PRODUCTION_BASELINE_LEGACY_A_NORMAL')
recommended = next(row for row in rows if row['scenario'] == 'RESEARCH_B_CAPPED')
{
    'total_overlap_reduction_pct': round(100 * (baseline['question_overlap'] - recommended['question_overlap']) / baseline['question_overlap'], 1),
    'single_overlap_reduction_pct': round(100 * (baseline['single_overlap'] - recommended['single_overlap']) / baseline['single_overlap'], 1),
    'likert_overlap_reduction_pct': round(100 * (baseline['likert_overlap'] - recommended['likert_overlap']) / baseline['likert_overlap'], 1),
    'semantic_overlap_reduction_pct': round(100 * (baseline['semantic_overlap'] - recommended['semantic_overlap']) / baseline['semantic_overlap'], 1),
}